In [0]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, broadcast, aggregate, sum as sparkSum, when, udf
from pyspark.sql.types import DoubleType


In [0]:
#loading original csv
df_raw = spark.read.format("csv")\
    .option("inferSchema", True)\
        .option("header", True)\
            .load("/FileStore/tables/fact_sales_data_v2.csv")

In [0]:
#loading dw fact table
df_processed = spark.read.format("delta")\
    .load("/FileStore/tables/FACT_Sales_day25")

In [0]:
#laoding dim tables
dim_product = spark.read.format("delta").load("/FileStore/tables/DIM_Product")
dim_store = spark.read.format("delta").load("/FileStore/tables/DIM_Store")
dim_employee = spark.read.format("delta").load("/FileStore/tables/DIM_Employee")

In [0]:
df_raw.printSchema()
df_processed.printSchema()

root
 |-- ProductCategory: string (nullable = true)
 |-- ProductName: string (nullable = true)
 |-- Brand: string (nullable = true)
 |-- StoreRegion: string (nullable = true)
 |-- StoreName: string (nullable = true)
 |-- StoreType: string (nullable = true)
 |-- SalesRep: string (nullable = true)
 |-- Department: string (nullable = true)
 |-- EmployeeRole: string (nullable = true)
 |-- UnitsSold: double (nullable = true)
 |-- UnitPrice: double (nullable = true)
 |-- Discount: double (nullable = true)
 |-- SaleDate: date (nullable = true)

root
 |-- Product_Id: long (nullable = true)
 |-- Store_Id: long (nullable = true)
 |-- Employee_Id: long (nullable = true)
 |-- UnitsSold: double (nullable = true)
 |-- UnitPrice: double (nullable = true)
 |-- Discount: double (nullable = true)
 |-- SaleDate: date (nullable = true)
 |-- NetRevenue: double (nullable = true)
 |-- Sales_Id: long (nullable = true)



In [0]:
#joining dim and fact

df_sales_processed = df_processed\
  .join(broadcast(dim_product), on= "Product_Id", how="left" ) \
    .join(broadcast(dim_employee), on = "Employee_Id", how = "left")\
      .join(broadcast(dim_store), on = "Store_Id", how="left")

In [0]:
#function to calculate net revenue
def netRev(UnitsSold, UnitPrice, Discount):
    if UnitsSold in (None, -1) or UnitPrice in (None, -1) or Discount in (None, -1):
        return None
    return UnitsSold * UnitPrice * (1 - Discount / 100)


In [0]:

#calculating netrevenue for raw data 
netRev_udf = udf(netRev, DoubleType())
df_raw_netRev = df_raw.withColumn(
    "NetRevenue",
    netRev_udf(
        col("UnitsSold"),
        col("UnitPrice"),
        col("Discount")
    )
)


In [0]:
#selecting datas where revenue not null for raw csv and not -1 for 
df_raw_netRev_clean = df_raw_netRev.where(col("NetRevenue").isNotNull())
df_sales_processed_clean = df_sales_processed.where(col("NetRevenue") > 0) 

In [0]:
df_sales_processed_clean.display()

Store_Id,Employee_Id,Product_Id,UnitsSold,UnitPrice,Discount,SaleDate,NetRevenue,Sales_Id,ProductCategory,ProductName,Brand,SalesRep,Department,EmployeeRole,StoreRegion,StoreName,StoreType
20,3,25,46.0,20.25,5.0,2022-10-14,884.925,5,Furniture,T-shirt,BrandC,Wendy Castillo,Home,Manager,East,StoreZ,Outlet
16,10,11,37.0,492.65,5.0,2024-05-06,17316.6475,7,Clothing,T-shirt,BrandC,John Harris,Home,Cashier,South,StoreY,Outlet
9,23,1,37.0,293.87,15.0,2023-04-04,9242.2115,8,Electronics,Smartphone,BrandC,Charles Fields,Home,Sales Associate,South,StoreX,Outlet
2,4,19,23.0,189.47,15.0,2022-12-26,3704.1385,9,Clothing,Jeans,BrandA,Wendy Castillo,Electronics,Manager,South,StoreY,Retail
10,2,23,25.0,359.08,10.0,2022-10-28,8079.3,10,Furniture,T-shirt,BrandB,Charles Fields,Apparel,Manager,East,StoreZ,Franchise
3,20,14,15.0,139.58,5.0,2023-11-09,1989.015,14,Furniture,Tablet,BrandC,Emily Vazquez,Apparel,Manager,East,StoreX,Retail
8,13,9,19.0,330.2,10.0,2022-12-16,5646.42,16,Electronics,Desk,BrandA,Emily Vazquez,Home,Manager,East,StoreX,Franchise
17,16,25,7.0,290.63,15.0,2022-07-28,1729.2485,17,Furniture,T-shirt,BrandC,Martha Long,Home,Cashier,West,StoreY,Franchise
1,5,4,7.0,349.27,10.0,2022-07-04,2200.401,18,Electronics,Desk,BrandC,Charles Fields,Home,Cashier,North,StoreY,Outlet
18,5,21,16.0,238.49,0.0,2024-08-27,3815.84,25,Furniture,Unknown,BrandA,Charles Fields,Home,Cashier,North,StoreZ,Franchise


In [0]:
# grouping by productCategory for fact_sales (removing all null values for netrevenue)
fact_sales_agg_report = df_sales_processed_clean.groupBy("ProductCategory").agg(
    sparkSum("UnitsSold").alias("TotalUnitsSold"),
    sparkSum("NetRevenue").alias("TotalRevenue")
)

#grouping by productCategory for raw csv (removing all null values for netrevenue)
raw_agg_report = df_raw_netRev_clean.groupBy("ProductCategory").agg(
    sparkSum("UnitsSold").alias("TotalUnitsSold"),
    sparkSum("NetRevenue").alias("TotalRevenue")
)

### Comparing both raw csv and cleaned warehouse data by comparing total unit sold an total revenue by grouping "Product Category"

In [0]:
fact_sales_agg_report.display()
raw_agg_report.display()

ProductCategory,TotalUnitsSold,TotalRevenue
Electronics,63.0,17089.0325
Clothing,60.0,21020.786
Furniture,146.0,29568.9485


ProductCategory,TotalUnitsSold,TotalRevenue
Electronics,63.0,17089.0325
Clothing,60.0,21020.786
Furniture,146.0,29568.9485


In [0]:
df_raw_netRev_clean.display()
df_raw_netRev_clean.display()

ProductCategory,ProductName,Brand,StoreRegion,StoreName,StoreType,SalesRep,Department,EmployeeRole,UnitsSold,UnitPrice,Discount,SaleDate,NetRevenue
Furniture,T-shirt,BrandC,East,StoreZ,Outlet,Wendy Castillo,Home,Manager,46.0,20.25,5.0,2022-10-14,884.925
Clothing,T-shirt,BrandC,South,StoreY,Outlet,John Harris,Home,Cashier,37.0,492.65,5.0,2024-05-06,17316.6475
Electronics,Smartphone,BrandC,South,StoreX,Outlet,Charles Fields,Home,Sales Associate,37.0,293.87,15.0,2023-04-04,9242.2115
Clothing,Jeans,BrandA,South,StoreY,Retail,Wendy Castillo,Electronics,Manager,23.0,189.47,15.0,2022-12-26,3704.1385
Furniture,T-shirt,BrandB,East,StoreZ,Franchise,Charles Fields,Apparel,Manager,25.0,359.08,10.0,2022-10-28,8079.3
Furniture,Tablet,BrandC,East,StoreX,Retail,Emily Vazquez,Apparel,Manager,15.0,139.58,5.0,2023-11-09,1989.015
Electronics,Desk,BrandA,East,StoreX,Franchise,Emily Vazquez,Home,Manager,19.0,330.2,10.0,2022-12-16,5646.42
Furniture,T-shirt,BrandC,West,StoreY,Franchise,Martha Long,Home,Cashier,7.0,290.63,15.0,2022-07-28,1729.2485
Electronics,Desk,BrandC,North,StoreY,Outlet,Charles Fields,Home,Cashier,7.0,349.27,10.0,2022-07-04,2200.401
Furniture,null,BrandA,North,StoreZ,Franchise,Charles Fields,Home,Cashier,16.0,238.49,0.0,2024-08-27,3815.84


ProductCategory,ProductName,Brand,StoreRegion,StoreName,StoreType,SalesRep,Department,EmployeeRole,UnitsSold,UnitPrice,Discount,SaleDate,NetRevenue
Furniture,T-shirt,BrandC,East,StoreZ,Outlet,Wendy Castillo,Home,Manager,46.0,20.25,5.0,2022-10-14,884.925
Clothing,T-shirt,BrandC,South,StoreY,Outlet,John Harris,Home,Cashier,37.0,492.65,5.0,2024-05-06,17316.6475
Electronics,Smartphone,BrandC,South,StoreX,Outlet,Charles Fields,Home,Sales Associate,37.0,293.87,15.0,2023-04-04,9242.2115
Clothing,Jeans,BrandA,South,StoreY,Retail,Wendy Castillo,Electronics,Manager,23.0,189.47,15.0,2022-12-26,3704.1385
Furniture,T-shirt,BrandB,East,StoreZ,Franchise,Charles Fields,Apparel,Manager,25.0,359.08,10.0,2022-10-28,8079.3
Furniture,Tablet,BrandC,East,StoreX,Retail,Emily Vazquez,Apparel,Manager,15.0,139.58,5.0,2023-11-09,1989.015
Electronics,Desk,BrandA,East,StoreX,Franchise,Emily Vazquez,Home,Manager,19.0,330.2,10.0,2022-12-16,5646.42
Furniture,T-shirt,BrandC,West,StoreY,Franchise,Martha Long,Home,Cashier,7.0,290.63,15.0,2022-07-28,1729.2485
Electronics,Desk,BrandC,North,StoreY,Outlet,Charles Fields,Home,Cashier,7.0,349.27,10.0,2022-07-04,2200.401
Furniture,null,BrandA,North,StoreZ,Franchise,Charles Fields,Home,Cashier,16.0,238.49,0.0,2024-08-27,3815.84


In [0]:
# grouping by Store region for fact_sales (removing all null values for netrevenue)
fact_sales_agg_report_by_region = df_sales_processed_clean.groupBy("StoreRegion").agg(
    sparkSum("UnitsSold").alias("TotalUnitsSold"),
    sparkSum("NetRevenue").alias("TotalRevenue")
)


#grouping by Store Region for raw csv (removing all null values for netrevenue)
raw_agg_report_by_region = df_raw_netRev_clean.groupBy("StoreRegion").agg(
    sparkSum("UnitsSold").alias("TotalUnitsSold"),
    sparkSum("NetRevenue").alias("TotalRevenue")
)

### Comparing both raw csv and cleaned warehouse data by comparing total unit sold an total revenue by grouping "Store Region"

In [0]:
fact_sales_agg_report_by_region.display()
raw_agg_report_by_region.display()

StoreRegion,TotalUnitsSold,TotalRevenue
South,134.0,43333.6175
East,105.0,16599.66
West,7.0,1729.2485
North,23.0,6016.241


StoreRegion,TotalUnitsSold,TotalRevenue
South,134.0,43333.6175
East,105.0,16599.66
West,7.0,1729.2485
North,23.0,6016.241


In [0]:
# grouping by By Sales Representative for fact_sales (removing all null values for netrevenue)
fact_sales_agg_report_by_salesRep = df_sales_processed_clean.groupBy("SalesRep").agg(
    sparkSum("UnitsSold").alias("TotalUnitsSold"),
    sparkSum("NetRevenue").alias("TotalRevenue")
)


#grouping by Sales Representative for raw csv (removing all null values for netrevenue)
raw_agg_report_by_salesRep = df_raw_netRev_clean.groupBy("SalesRep").agg(
    sparkSum("UnitsSold").alias("TotalUnitsSold"),
    sparkSum("NetRevenue").alias("TotalRevenue")
)

### Comparing both raw csv and cleaned warehouse data by comparing total unit sold an total revenue by grouping "Sales representative"

In [0]:
fact_sales_agg_report_by_salesRep.display()
raw_agg_report_by_salesRep.display()

SalesRep,TotalUnitsSold,TotalRevenue
Wendy Castillo,69.0,4589.0635
John Harris,37.0,17316.6475
Emily Vazquez,34.0,7635.435
Billy Perez,37.0,13070.62
Charles Fields,85.0,23337.7525
Martha Long,7.0,1729.2485


SalesRep,TotalUnitsSold,TotalRevenue
Wendy Castillo,69.0,4589.0635
John Harris,37.0,17316.6475
Emily Vazquez,34.0,7635.435
Billy Perez,37.0,13070.62
Charles Fields,85.0,23337.7525
Martha Long,7.0,1729.2485
